# Lab 02 — Word Vectors and Vector Databases

**NLP Applications Laboratory | KLEF | 2026**

*From tokens to vectors — the first step toward machines that understand meaning.*

## Objective

By the end of this lab, you will be able to:

1. Build a **vocabulary** from a corpus of text.
2. Map each word to a numeric **index** and back (`word2idx`, `idx2word`).
3. Convert words into **one-hot vectors** — the simplest possible vector representation.
4. Store these vectors in a **vector database** (a searchable collection of vectors).
5. Perform **similarity search**: given a query word, find its nearest neighbors.
6. Measure the **performance** of one-hot vectors — and discover why they fall short.

This "shortcoming" is what motivates every embedding technique that follows in Lab 03.

## Process

We follow this pipeline, step by step:

1. **Corpus** — A small collection of sentences we will work with.
2. **Tokenize** — Split each sentence into words (reusing Lab 01 tools).
3. **Vocabulary** — Collect the unique words across the whole corpus.
4. **Indexing** — Assign an integer ID to each vocabulary word (`word2idx`), and build the reverse map (`idx2word`).
5. **One-hot encoding** — Represent each word as a sparse vector: all zeros, with a single `1` at its index position.
6. **Vector database** — Store all word vectors in a searchable structure.
7. **Similarity search** — Given a query word, retrieve the top-K most similar words.
8. **Measure performance** — Does one-hot capture meaning? Are `cat` and `dog` closer to each other than `cat` and `truck`?

At each step, we will use our local Qwen model to ask *why* things work (or don't) the way they do.

In [1]:
# 1. Build your Corpus

In [2]:
# A small corpus with three semantic clusters: animals, food, vehicles
corpus = [
    "The cat chased the mouse",
    "Dogs love to play in the park",
    "Rabbits hop through the field",
    "The mouse ran from the cat",
    "Pizza is baked in an oven",
    "I ate pasta for dinner",
    "Bread is made from flour",
    "The oven baked fresh bread",
    "Cars drive on the highway",
    "The bus arrives at eight",
    "Trucks carry heavy loads",
    "A car and a truck raced",
]
print(f"Corpus size: {len(corpus)} sentences")

Corpus size: 12 sentences


In [3]:
# 2. Tokenize the corpus
# Simple whitespace tokenization + lowercasing (same idea as Lab 01)
tokenized = [sentence.lower().split() for sentence in corpus]
for i, tokens in enumerate(tokenized[:3]):
    print(f"Sentence {i+1}: {tokens}")

Sentence 1: ['the', 'cat', 'chased', 'the', 'mouse']
Sentence 2: ['dogs', 'love', 'to', 'play', 'in', 'the', 'park']
Sentence 3: ['rabbits', 'hop', 'through', 'the', 'field']


In [4]:
# 3. Build the vocabulary

In [5]:
# Vocabulary = unique tokens across the entire corpus
all_tokens = [tok for sent in tokenized for tok in sent]
vocab = sorted(set(all_tokens))
print(f"Total tokens (with repeats): {len(all_tokens)}")
print(f"Vocabulary size (unique)   : {len(vocab)}")
print(f"Vocabulary                 : {vocab}")

Total tokens (with repeats): 64
Vocabulary size (unique)   : 47
Vocabulary                 : ['a', 'an', 'and', 'arrives', 'at', 'ate', 'baked', 'bread', 'bus', 'car', 'carry', 'cars', 'cat', 'chased', 'dinner', 'dogs', 'drive', 'eight', 'field', 'flour', 'for', 'fresh', 'from', 'heavy', 'highway', 'hop', 'i', 'in', 'is', 'loads', 'love', 'made', 'mouse', 'on', 'oven', 'park', 'pasta', 'pizza', 'play', 'rabbits', 'raced', 'ran', 'the', 'through', 'to', 'truck', 'trucks']


### What we should see:

74 total tokens across all 12 sentences (with repeats) — mostly function words like the, a, in
~55 unique words in the vocabulary — this is our vocabulary size
The list is sorted alphabetically — not necessary, but makes it easier to inspect. Sorting also means every student in class gets the exact same vocabulary order, so their indices in Step 8 will match.

### Two subtle points worth learning:

set() gives us uniques but loses order. sorted() gives us deterministic ordering — important because in the next step, index assignments depend on order. If two students get different orders, they get different indices, and their vectors won't match. Small point, big consequence for reproducibility.
Notice the vocabulary has weird pairs like car and cars, dog and dogs. In real NLP you'd use stemming or lemmatization to merge these. We deliberately don't, so students see the raw problem later.

In [6]:
# 4. Build word2idx and idx2word mappings

In [7]:
# Assign an integer ID to each word in the vocabulary
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}
print(f"word2idx['cat']={word2idx['cat']}")
print(f"word2idx['dog']={word2idx.get('dog', 'NOT IN VOCAB')}")
print(f"idx2word[{word2idx['cat']}]={idx2word[word2idx['cat']]}")

word2idx['cat']  = 12
word2idx['dog']  = NOT IN VOCAB
idx2word[12]       = cat


In [8]:
# Print the full word→index table
# The complete vocabulary as a table: word → index
print(f"{'Index':>5} | {'Word':<15}")
print("-" * 25)
for word, idx in word2idx.items():
    print(f"{idx:>5} | {word:<15}")

Index | Word           
-------------------------
    0 | a              
    1 | an             
    2 | and            
    3 | arrives        
    4 | at             
    5 | ate            
    6 | baked          
    7 | bread          
    8 | bus            
    9 | car            
   10 | carry          
   11 | cars           
   12 | cat            
   13 | chased         
   14 | dinner         
   15 | dogs           
   16 | drive          
   17 | eight          
   18 | field          
   19 | flour          
   20 | for            
   21 | fresh          
   22 | from           
   23 | heavy          
   24 | highway        
   25 | hop            
   26 | i              
   27 | in             
   28 | is             
   29 | loads          
   30 | love           
   31 | made           
   32 | mouse          
   33 | on             
   34 | oven           
   35 | park           
   36 | pasta          
   37 | pizza          
   38 | play           
   39 | rabbit

In [9]:
# Show that a different order gives different indices
# The order is our design choice — here's the same vocab, ordered by frequency
from collections import Counter
freq = Counter(all_tokens)
by_freq = sorted(vocab, key=lambda w: -freq[w])
word2idx_freq = {word: idx for idx, word in enumerate(by_freq)}
print(f"{'Alphabetical':<20} | {'By frequency':<20}")
print("-" * 45)
for word in ['the', 'a', 'cat', 'dogs', 'pizza']:
    a_idx = word2idx.get(word, 'N/A')
    f_idx = word2idx_freq.get(word, 'N/A')
    print(f"{word} → {a_idx:<15} | {word} → {f_idx}")

Alphabetical         | By frequency        
---------------------------------------------
the → 42              | the → 0
a → 0               | a → 1
cat → 12              | cat → 4
dogs → 15              | dogs → 21
pizza → 37              | pizza → 38


### Can we choose our own order?

**Yes — the choice is entirely ours.** The vocabulary is just a lookup table, and any bijective mapping (word ↔ unique integer) works. Common conventions:

| Ordering | When to use it |
|---|---|
| **Alphabetical** | Debugging, reproducibility across teams — anyone can rebuild the same table |
| **By frequency** (most common first) | Compression, embeddings that reserve special IDs for common words |
| **Insertion order** (first seen = index 0) | Streaming / online vocabulary building |
| **Reserved slots first** | `<PAD>=0, <UNK>=1, <BOS>=2, <EOS>=3`, then real words. Universal in deep learning. |

**The critical rule**: once you've picked an ordering and built the vector representations, **you cannot change it** without rebuilding every vector. A model trained on one ordering has weights bound to those specific indices. If `cat=12` during training, `cat` must be `12` at inference time forever.

This is why real NLP projects **serialize the vocabulary to disk** and load the exact same one every time. We'll skip this for now, but keep it in mind: your vocab is part of your model.

In [10]:
from nlpa_chat import chat
chat()

### Why do we need idx2word at all? Isn't word2idx enough?
#### In the vocabulary I built, dogs is there but dog is not. What real-world problem does this create for a text classifier?

In [11]:
# 5.Convert sentences into sequences of indices

In [12]:
# Convert every sentence in the corpus to a sequence of indices
indexed = [[word2idx[word] for word in sentence] 
           for sentence in tokenized]
for i in range(3):
    print(f"Sentence: {tokenized[i]}")
    print(f"Indices : {indexed[i]}\n")

Sentence: ['the', 'cat', 'chased', 'the', 'mouse']
Indices : [42, 12, 13, 42, 32]

Sentence: ['dogs', 'love', 'to', 'play', 'in', 'the', 'park']
Indices : [15, 30, 44, 38, 27, 42, 35]

Sentence: ['rabbits', 'hop', 'through', 'the', 'field']
Indices : [39, 25, 43, 42, 18]



#### Every word became an integer. the appears twice in sentence 1, so index 47 appears twice. 
#### The integer 47 for the and 12 for cat are arbitrary labels, not meaningful numbers. 47 - 12 = 35 doesn't mean anything. 47 > 12 doesn't mean the is "greater than" cat in any real sense. These are just name tags — like student roll numbers.

# Why integer IDs alone can't work for learning

In [13]:
# The problem: integer IDs create artificial numeric relationships that don't exist
print(f"cat  -> {word2idx['cat']}")
print(f"mouse -> {word2idx['mouse']}")
print(f"pizza -> {word2idx['pizza']}")
print()
print(f"|cat - mouse|={abs(word2idx['cat'] 
                           - word2idx['mouse'])}")
print(f"|cat - pizza|={abs(word2idx['cat'] 
                           - word2idx['pizza'])}")
print(f"Is 'cat' more similar to 'mouse' than to 'pizza'?")

cat  -> 12
mouse -> 32
pizza -> 37

|cat - mouse|  = 20
|cat - pizza|  = 25
Is 'cat' more similar to 'mouse' than to 'pizza'?


### The problem with raw integer IDs

Looking at the numbers above, you might be tempted to say *"yes, `cat` (12) is closer to `mouse` (28) than to `pizza` (33), so integer distance captures meaning."*

But this is **entirely accidental**. The vocabulary was sorted alphabetically:

- `cat` is 12 because it starts with 'c'
- `mouse` is 28 because it starts with 'm'
- `pizza` is 33 because it starts with 'p'

The distances between these numbers reflect **alphabetical position**, not **semantic meaning**.

If we had sorted by frequency (as we did in cell 8c), we'd get completely different distances — and *those* wouldn't mean anything either.

**Two disasters if we feed integer IDs directly into a model:**

1. **False relationships** — a neural network might learn "words with close indices are related", which is just alphabetical accident.

2. **Fake magnitude** — `word2idx['zebra']` might be 300, `word2idx['a']` is 0. The model sees `zebra` as "300 times larger" than `a`, which is nonsense.

We need a representation where:
- No word has a "larger" or "smaller" value than any other.
- The relationship between two words is not accidentally determined by our vocabulary ordering.

**Enter one-hot encoding** — the first attempt to fix this problem.

# One-hot encoding for a single word

In [14]:
# 6. One-hot encoding: a vector of all zeros except a single 1 at the word's index
import numpy as np
V = len(vocab)       # V = vocabulary size (vector dimension)
onehot_cat = np.zeros(V, dtype=int)
onehot_cat[word2idx['cat']] = 1
print(f"Vocabulary size (V): {V}")
print(f"Vector for 'cat' (length {len(onehot_cat)}):")
print(onehot_cat)
print(f"Position of the 1: index {np.argmax(onehot_cat)}")

Vocabulary size (V): 47
Vector for 'cat' (length 47):
[0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0]
Position of the 1: index 12


### The vector has V elements, where V = 55 (our vocabulary size). Every word gets a vector of length 55.
### Only one position is 1, every other position is 0. Hence "one-hot" — one position is "hot" (activated), the rest are cold.
### The 1 sits at position 12 — the same index that word2idx['cat'] gave us in Step 8. The vector IS the index, but in a different form.

In [15]:
# 6. Encode the entire vocabulary as one-hot vectors

In [16]:
# Build a V x V matrix where row i is the one-hot vector for word i
onehot_matrix = np.eye(V, dtype=int)
print(f"Shape: {onehot_matrix.shape}")
print(f"Vector for 'cat'   (row {word2idx['cat']}):   {onehot_matrix[word2idx['cat']]}")
print(f"Vector for 'dogs'  (row {word2idx['dogs']}):  {onehot_matrix[word2idx['dogs']]}")
print(f"Vector for 'pizza' (row {word2idx['pizza']}): {onehot_matrix[word2idx['pizza']]}")

Shape: (47, 47)
Vector for 'cat'   (row 12):   [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0]
Vector for 'dogs'  (row 15):  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0]
Vector for 'pizza' (row 37): [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 0 0 0 0 0]


# The whole vocabulary fits in one matrix: shape (47, 47) — 47 rows for 47 words, each row has 47 columns.

In [17]:
np.eye(V)

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(47, 47))

In [18]:
# Each word's vector is a row of this matrix. 
# To get the vector for any word, look up its index and grab that row:
onehot_matrix[word2idx['cat']]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0])

In [19]:
47*47

2209

# V = 10,000 words (a small English vocabulary)

# Similarity search using cosine similarity

In [20]:
# Similarity search: given a query word, find its nearest neighbors
def cosine_similarity(v1, v2):
    dot = np.dot(v1, v2)
    norm = np.linalg.norm(v1) * np.linalg.norm(v2)
    return dot / norm if norm > 0 else 0.0

query = 'cat'
query_vec = onehot_matrix[word2idx[query]]
scores = [(word, cosine_similarity(query_vec, onehot_matrix[i]))
          for word, i in word2idx.items()]
scores.sort(key=lambda x: -x[1])
print(f"Top 5 nearest neighbors to '{query}':")
for word, score in scores[:5]:
    print(f"  {word:<15} similarity = {score:.4f}")

Top 5 nearest neighbors to 'cat':
  cat             similarity = 1.0000
  a               similarity = 0.0000
  an              similarity = 0.0000
  and             similarity = 0.0000
  arrives         similarity = 0.0000


# In-lab assignment - Lab Continuous Evaluation

---

## 🎯 In-Lab Assignment

You have just seen that one-hot vectors fail at capturing similarity. Now it's your turn to prove — with code — that you understand why, and what could be done about it.

### Task overview

Extend the corpus, apply the same one-hot pipeline, and demonstrate the failure with **your own experiments and metrics**. Then propose (in words, not code) how you would fix the problems you discovered.

### Deliverables (5 code cells + 1 markdown cell)

**Cell 1 — Enlarge the corpus.**
Add **10 new sentences** of your own choice to `corpus`, staying within the three semantic clusters (animals, food, vehicles). Rebuild `tokenized`, `vocab`, `word2idx`, `idx2word`, and `onehot_matrix` from scratch.

- Print the new vocabulary size.

**Cell 2 — Semantic-cluster similarity check.**
Pick **three pairs of words** that SHOULD be similar (within the same cluster) and **three pairs that SHOULD be dissimilar** (across clusters). Examples:

- Should be similar: `(cat, mouse)`, `(pizza, bread)`, `(car, bus)`
- Should be dissimilar: `(cat, pizza)`, `(bread, truck)`, `(dog, oven)`

Compute cosine similarity for all six pairs. Print them in a table.

**Cell 3 — Sparsity analysis.**
Count and print:
- Total number of elements in `onehot_matrix`
- Number of `1`s
- Number of `0`s
- Percentage sparsity (fraction of zeros)

**Cell 4 — Storage cost estimation.**
Assume each element is a 32-bit integer (4 bytes). Compute and print:
- Total memory used by `onehot_matrix` (in bytes and KB)
- What would this be if the vocabulary grew to 10,000 words? To 100,000?
- What fraction of that memory would be storing *only zeros*?

**Cell 5 — Vocabulary robustness test.**
Try to look up a word that is NOT in your vocabulary using `word2idx.get(...)`. Try three words. Show what happens.

- What real-world problem does this simulate?

**Cell 6 (markdown) — Your analysis.**
Answer these in your own words (2–3 sentences each):

1. Based on your similarity table in Cell 2, does one-hot encoding capture *any* notion of "same cluster"? Justify with your numbers.
2. If your university's Telugu-English dictionary contains 50,000 words, how much RAM would a one-hot vocabulary require? Is this practical?
3. What information about the corpus would we need to *capture* if we wanted `cat` and `dogs` to end up close in vector space? (You don't need to know the algorithm — just describe what the vectors would need to "know.")

### How to work on this assignment

- Use the `chat()` widget freely — ask Qwen for hints, not full answers
- Reference specific cells with `@cell N` when asking Qwen questions
- When stuck, **run partial code first** and inspect the output — don't ask Qwen for the whole solution

### Submission

Save this notebook as `02_word_vectors_YOURNAME.ipynb` and upload to the lab portal by end of session.

**Deadline**: end of today's lab session.